# Part 1: Data Preparation
**LAB | Evaluation Dashboard** · Ayhan Demir · Tool: Tableau

Data choice: **Option B**, sample evaluation data generated with the lab's script.

How to run: keep this notebook inside the folder `lab_evaluation_dashboard_ayhan_demir/` and run the cells top to bottom. Every file is written to that folder.

## Step 1: Choose your data source (Option B)
The script below is the lab's generator script, unchanged. It writes `evaluation_data.csv`.

In [6]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Set random seed for reproducibility
np.random.seed(42)

# Define evaluation categories
categories = ['reasoning', 'knowledge', 'code', 'instruction_following', 'tool_calling']

# Define model versions
model_versions = ['gpt-4', 'gpt-4-turbo', 'gpt-3.5-turbo', 'claude-3-opus']

# Generate date range (last 90 days)
start_date = datetime.now() - timedelta(days=90)
dates = [start_date + timedelta(days=i) for i in range(90)]

# Generate evaluation data
data = []
evaluation_id = 1

for date in dates:
    for category in categories:
        # Generate 5-15 evaluations per category per day
        num_evaluations = np.random.randint(5, 16)

        for _ in range(num_evaluations):
            model = np.random.choice(model_versions)

            # Generate scores with category-specific distributions
            if category == 'reasoning':
                # Reasoning tends to have lower scores, more variance
                score = np.random.beta(2, 3) * 100
            elif category == 'knowledge':
                # Knowledge tends to have higher scores
                score = np.random.beta(4, 2) * 100
            elif category == 'code':
                # Code has medium scores with high variance
                score = np.random.beta(3, 3) * 100
            elif category == 'instruction_following':
                # Instruction following tends to be high
                score = np.random.beta(5, 2) * 100
            else:  # tool_calling
                # Tool calling has medium scores
                score = np.random.beta(3, 3) * 100

            # Add some noise
            score = max(0, min(100, score + np.random.normal(0, 5)))

            data.append({
                'evaluation_id': evaluation_id,
                'category': category,
                'score': round(score, 2),
                'model_version': model,
                'date': date.strftime('%Y-%m-%d')
            })
            evaluation_id += 1

# Create DataFrame
df = pd.DataFrame(data)

# Save to CSV
df.to_csv('evaluation_data.csv', index=False)

print(f"Generated {len(df)} evaluation records")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"\nScore summary by category:")
print(df.groupby('category')['score'].describe())
print(f"\nFirst few rows:")
print(df.head())

Generated 4489 evaluation records
Date range: 2026-08-26 to 2026-11-23

Score summary by category:
                       count       mean        std    min      25%     50%  \
category                                                                     
code                   914.0  49.617757  19.833349   0.73  35.5425  49.465   
instruction_following  898.0  72.205056  16.722377  21.32  60.3975  74.510   
knowledge              871.0  65.984742  19.110877   6.83  52.6000  68.490   
reasoning              879.0  40.847691  20.697335   0.00  25.3000  39.250   
tool_calling           927.0  48.346677  19.292148   1.66  34.1200  47.910   

                           75%     max  
category                                
code                   64.1750   97.19  
instruction_following  85.5625  100.00  
knowledge              80.7800  100.00  
reasoning              54.6050   96.48  
tool_calling           62.0550   95.34  

First few rows:
   evaluation_id   category  score  model_version 

## Step 2: Clean and format data
Checks from the brief: numeric scores, consistent categories, YYYY-MM-DD dates, no missing score or category. Then save `evaluation_data_clean.csv` (UTF-8, comma).

In [7]:
# Checks on the raw file
print("PASS score is numeric" if pd.api.types.is_numeric_dtype(raw["score"]) else "FAIL score is text")
print("PASS categories consistent" if set(raw["category"]) == set(categories) else f"FAIL categories: {set(raw['category'])}")
print("PASS dates YYYY-MM-DD" if raw["date"].str.match(r"^\d{4}-\d{2}-\d{2}$").all() else "FAIL date format")
print("Missing score or category:", raw[["score", "category"]].isna().sum().sum())

PASS score is numeric
PASS categories consistent
PASS dates YYYY-MM-DD
Missing score or category: 0


In [8]:
# Clean: drop rows missing critical data, tidy text, keep scores in 0-100
clean = raw.dropna(subset=["score", "category"]).copy()
clean["category"] = clean["category"].str.strip().str.lower()
clean["model_version"] = clean["model_version"].str.strip()
clean["score"] = pd.to_numeric(clean["score"]).clip(0, 100)
clean["date"] = pd.to_datetime(clean["date"]).dt.strftime("%Y-%m-%d")

# Save the clean file for Tableau
clean.to_csv("evaluation_data_clean.csv", index=False, encoding="utf-8")

# Re-check the saved file
check = pd.read_csv("evaluation_data_clean.csv")
print("PASS no rows lost" if len(check) == len(raw) else f"Rows removed: {len(raw) - len(check)}")
print("PASS scores between 0 and 100" if check["score"].between(0, 100).all() else "FAIL score range")
print("PASS no missing values" if check.isna().sum().sum() == 0 else "FAIL missing values")

PASS no rows lost
PASS scores between 0 and 100
PASS no missing values


## Step 3: Prepare data for import
Transformations noted for Tableau (done in the tool, not in the file):
- Score bins of size 10 for the histogram (Tableau: Create > Bins)
- Average aggregation for score in the comparison chart
- No filters at import: filters are added on the dashboard

In [9]:
# Final review: row count, column names, data types, file path
if len(check) >= 100:
    print("PASS rows:", len(check))
else:
    print("FAIL fewer than 100 rows")
print("\nColumns and types:")
print(check.dtypes)

# What Tableau should detect in Step 4
tableau_types = pd.DataFrame({
    "column": expected_cols,
    "Tableau role": ["Dimension (convert if shown as measure)", "Dimension", "Measure", "Dimension", "Dimension"],
    "type icon": ["#", "Abc", "#", "Abc", "calendar"],
})
print("\n", tableau_types.to_string(index=False))

print("\nFile path for Tableau:", Path("evaluation_data_clean.csv").resolve())

PASS rows: 4489

Columns and types:
evaluation_id      int64
category             str
score            float64
model_version        str
date                 str
dtype: object

        column                            Tableau role type icon
evaluation_id Dimension (convert if shown as measure)         #
     category                               Dimension       Abc
        score                                 Measure         #
model_version                               Dimension       Abc
         date                               Dimension  calendar

File path for Tableau: /Users/macbook-air/Desktop/AI consulting/Week8/Day1/lab-evaluation-dashboard/lab_evaluation_dashboard_ayhan_demir/evaluation_data_clean.csv


In [10]:
# Deliverable: data_source.md (which option and why)
text = f"""# Data Source

**Option B: generated sample data.** I used the lab's generator script (seed 42) to create `evaluation_data.csv` and cleaned it into `evaluation_data_clean.csv` ({len(check)} evaluations, 5 categories, 4 model versions, {check['date'].min()} to {check['date'].max()}).
I chose Option B because my earlier lab outputs do not hold one file with categories, model versions and 90 days of dates together, which this dashboard needs for its filters.
The data is synthetic: scores follow fixed category patterns and the model version is assigned at random, so the dashboard shows the method, not real model performance.
"""
Path("data_source.md").write_text(text, encoding="utf-8")
print(text)

# Data Source

**Option B: generated sample data.** I used the lab's generator script (seed 42) to create `evaluation_data.csv` and cleaned it into `evaluation_data_clean.csv` (4489 evaluations, 5 categories, 4 model versions, 2026-08-26 to 2026-11-23).
I chose Option B because my earlier lab outputs do not hold one file with categories, model versions and 90 days of dates together, which this dashboard needs for its filters.
The data is synthetic: scores follow fixed category patterns and the model version is assigned at random, so the dashboard shows the method, not real model performance.

